# Homework 4 - Instrumental Variables and Regression Discontinuity

**Part A (Questions 1-2).** `homework_4.1.csv`. `X` is the treatment, `W` the confounder, `Y` the
outcome, `Z` the instrument. The effect is estimated two ways:

1. **Pooled Wald ratio** - subtract the average `Y` for `Z = 1` and `Z = 0`, subtract the average
   `X` for `Z = 1` and `Z = 0`, divide.
   $$\hat{\tau} = \frac{\bar{Y}_{Z=1} - \bar{Y}_{Z=0}}{\bar{X}_{Z=1} - \bar{X}_{Z=0}}$$
2. **Stratified Wald ratio** - do the same inside narrow bands of `W`, then average the per-band
   ratios across all the bands.

**Part B (Questions 3-6).** `homework_4.2.a` and `homework_4.2.b`, test score `X`, cutoff **80**,
outcome `Y` (got into college). On each side of the cutoff `Y` is fit as a straight line in `X`,
which gives the jump at the cutoff and the slope on either side.

| # | Question | Where |
|---|---|---|
| 1 | Effect from the pooled Wald ratio | Section 3 |
| 2 | Effect from the ratio averaged over narrow bands of `W` | Section 4 |
| 3 | Does the math course help, in each dataset | Section 7 |
| 4 | Which dataset has a nonzero linear term | Section 8 |
| 5 | Is `Y` increasing or decreasing before the cutoff | Section 8 |
| 6 | Is the slope higher or lower after the cutoff | Section 8 |

**Why the ratio works.** The instrument `Z` shifts `X` without touching `Y` except through `X`.
So the part of the `Y` gap that `Z` opens up is the effect multiplied by the part of the `X` gap
that `Z` opens up, and dividing one by the other recovers the effect.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

CSV_IV = "homework_4.1.csv"          # Part A
CUTOFF = 80                          # Part B cutoff score
N_BINS = 10                          # number of narrow W bands for Question 2

pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

def find_file(*candidates):
    # The 4.2 files are referred to without an extension, so try the usual spellings.
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"none of these exist: {candidates}")

# Part A - Instrumental variable (Questions 1-2)

## 2. Load `homework_4.1.csv`

In [ ]:
iv = pd.read_csv(CSV_IV)

print("shape:", iv.shape)
print("\nmissing values:\n", iv.isna().sum(), sep="")
iv.head()

In [ ]:
def pick(df, letter, fallback_pool):
    hit = next((c for c in df.columns if c.strip().upper() == letter), None)
    return hit if hit is not None else fallback_pool.pop(0)

pool = list(iv.columns)
XT = pick(iv, "X", pool)   # treatment
WC = pick(iv, "W", pool)   # confounder
YO = pick(iv, "Y", pool)   # outcome
ZI = pick(iv, "Z", pool)   # instrument

print(f"treatment  X -> {XT}")
print(f"confounder W -> {WC}")
print(f"outcome    Y -> {YO}")
print(f"instrument Z -> {ZI}")

d = iv[[XT, WC, YO, ZI]].dropna().reset_index(drop=True)
print(f"\nn = {len(d)}")
print(f"{ZI} takes values: {sorted(d[ZI].unique())[:10]}")
print(f"{XT} takes values: {sorted(d[XT].unique())[:10]}")
d.groupby(ZI)[[XT, WC, YO]].agg(["mean", "count"])

## 3. Question 1 - the pooled Wald ratio

One `Y` gap and one `X` gap across the whole sample, then the ratio.

In [ ]:
z1 = d[d[ZI] == 1]
z0 = d[d[ZI] == 0]

dY = z1[YO].mean() - z0[YO].mean()
dX = z1[XT].mean() - z0[XT].mean()
wald_pooled = dY / dX

print(f"average {YO}:  Z=1 {z1[YO].mean():.6f}   Z=0 {z0[YO].mean():.6f}   difference {dY:.6f}")
print(f"average {XT}:  Z=1 {z1[XT].mean():.6f}   Z=0 {z0[XT].mean():.6f}   difference {dX:.6f}")
print(f"\n>>> Q1  effect = {dY:.6f} / {dX:.6f} = {wald_pooled:.6f}")

In [ ]:
# Cross-checks. Two-stage least squares gives the identical number for a binary instrument,
# and the plain OLS of Y on X shows how much the confounder distorts things.
def ols_coef(y, *regressors):
    X = np.column_stack([np.ones(len(y))] + [np.asarray(r, float) for r in regressors])
    beta, *_ = np.linalg.lstsq(X, np.asarray(y, float), rcond=None)
    return beta

x_hat = ols_coef(d[XT], d[ZI])                      # first stage: X on Z
fitted_x = x_hat[0] + x_hat[1] * d[ZI].to_numpy(float)
tsls = ols_coef(d[YO], fitted_x)[1]                 # second stage: Y on fitted X

print(f"2SLS estimate                         : {tsls:.6f}")
print(f"pooled Wald ratio                     : {wald_pooled:.6f}  "
      f"(match: {np.isclose(tsls, wald_pooled)})")
print(f"naive OLS of {YO} on {XT} (biased)         : {ols_coef(d[YO], d[XT])[1]:.6f}")
print(f"OLS of {YO} on {XT} controlling for {WC}     : {ols_coef(d[YO], d[XT], d[WC])[1]:.6f}")
print(f"\nfirst stage: {ZI} moves {XT} by {x_hat[1]:.6f} (a weak instrument would make the "
      f"ratio unstable)")

## 4. Question 2 - the ratio averaged over narrow bands of `W`

`W` is cut into equal-sized bands. Inside each band the same `Y` gap and `X` gap are computed
between `Z = 1` and `Z = 0`, giving one ratio per band, and those ratios are averaged.

A band is dropped when it has no rows on one side of the instrument, or when its `X` gap is
essentially zero (dividing by a near-zero denominator produces a meaningless spike).

In [ ]:
def stratified_wald(data, n_bins, min_dx=1e-3, weighted=False, verbose=True):
    # Cut W into equal-count bands, take the Wald ratio inside each, then average.
    work = data.copy()
    work["W_band"] = pd.qcut(work[WC], q=n_bins, duplicates="drop")

    rows = []
    for band, g in work.groupby("W_band", observed=True):
        g1, g0 = g[g[ZI] == 1], g[g[ZI] == 0]
        if len(g1) == 0 or len(g0) == 0:
            rows.append({"band": band, "n": len(g), "n_z1": len(g1), "n_z0": len(g0),
                         "dY": np.nan, "dX": np.nan, "ratio": np.nan, "used": False})
            continue
        dy = g1[YO].mean() - g0[YO].mean()
        dx = g1[XT].mean() - g0[XT].mean()
        ok = abs(dx) > min_dx
        rows.append({"band": band, "n": len(g), "n_z1": len(g1), "n_z0": len(g0),
                     "dY": dy, "dX": dx, "ratio": dy / dx if ok else np.nan, "used": ok})

    table = pd.DataFrame(rows)
    good = table[table["used"]]
    simple = good["ratio"].mean()
    wtd = np.average(good["ratio"], weights=good["n"]) if len(good) else np.nan

    if verbose:
        print(f"{len(good)} of {len(table)} bands usable")
        print(f"simple average of the band ratios   : {simple:.6f}")
        print(f"size-weighted average               : {wtd:.6f}")
        print(f"median band ratio                   : {good['ratio'].median():.6f}")
    return (wtd if weighted else simple), table

wald_stratified, band_table = stratified_wald(d, N_BINS)
print(f"\n>>> Q2  effect averaged over {N_BINS} bands of {WC} = {wald_stratified:.6f}")
band_table

In [ ]:
# The answer moves with the number of bands, so check a range of them before choosing.
sens = []
for k in [2, 4, 5, 8, 10, 15, 20, 25, 50]:
    simple, tbl = stratified_wald(d, k, verbose=False)
    wtd, _ = stratified_wald(d, k, weighted=True, verbose=False)
    sens.append({"n_bins": k, "bands_used": int(tbl["used"].sum()),
                 "simple_average": simple, "weighted_average": wtd})

sens = pd.DataFrame(sens).set_index("n_bins")
print(f"pooled Wald ratio for reference: {wald_pooled:.6f}\n")
sens

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
good = band_table[band_table["used"]]
pos = np.arange(len(good))
axes[0].bar(pos, good["ratio"], color="steelblue", alpha=0.85)
axes[0].axhline(wald_stratified, color="crimson", lw=2,
                label=f"average of bands {wald_stratified:.3f}")
axes[0].axhline(wald_pooled, color="darkgreen", lw=2, ls="--",
                label=f"pooled ratio {wald_pooled:.3f}")
axes[0].set_xticks(pos)
axes[0].set_xticklabels([f"{i+1}" for i in pos])
axes[0].set_xlabel(f"{WC} band (low to high)"); axes[0].set_ylabel("Wald ratio in the band")
axes[0].set_title("Effect estimated inside each band")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(sens.index, sens["simple_average"], "o-", label="simple average of bands")
axes[1].plot(sens.index, sens["weighted_average"], "s-", label="size-weighted")
axes[1].axhline(wald_pooled, color="darkgreen", ls="--", label="pooled ratio")
axes[1].set_xlabel("number of bands"); axes[1].set_ylabel("estimated effect")
axes[1].set_title("Sensitivity to how narrow the bands are")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Part B - Regression discontinuity (Questions 3-6)

## 5. Load the two datasets

Both files are read, the score column and the outcome column identified, and everything below is
computed for each dataset separately.

In [ ]:
path_a = find_file("homework_4.2.a.csv", "homework_4.2.a", "homework_4.2a.csv", "homework_4.2a")
path_b = find_file("homework_4.2.b.csv", "homework_4.2.b", "homework_4.2b.csv", "homework_4.2b")
print("dataset a:", path_a)
print("dataset b:", path_b)

def load_rd(path):
    df = pd.read_csv(path)
    pool = list(df.columns)
    xc = next((c for c in df.columns if c.strip().upper() == "X"), None)
    yc = next((c for c in df.columns if c.strip().upper() == "Y"), None)
    if xc is None:
        xc = [c for c in pool if c != yc][0]
    if yc is None:
        yc = [c for c in pool if c != xc][0]
    out = df[[xc, yc]].dropna().rename(columns={xc: "score", yc: "outcome"})
    return out.sort_values("score").reset_index(drop=True)

rd = {"a": load_rd(path_a), "b": load_rd(path_b)}

for k, v in rd.items():
    print(f"\ndataset {k}: n = {len(v)}, "
          f"score range {v['score'].min():.3f} to {v['score'].max():.3f}, "
          f"below cutoff {int((v['score'] < CUTOFF).sum())}, "
          f"at/above {int((v['score'] >= CUTOFF).sum())}")
    print(f"  outcome values: {sorted(v['outcome'].unique())[:6]}"
          f"{' ...' if v['outcome'].nunique() > 6 else ''}")
rd["a"].head()

## 6. Fit a line on each side of the cutoff

For each dataset, `Y` is regressed on the centred score $X - 80$ separately below and at/above
the cutoff. The intercepts give the fitted value on each side *at* the cutoff, so their
difference is the jump; the slopes answer Questions 5 and 6.

The equivalent single regression is
$$Y = \beta_0 + \beta_1 D + \beta_2 (X - 80) + \beta_3 D\,(X - 80),$$
with $D = 1$ when $X \ge 80$. There $\beta_1$ is the jump and $\beta_3$ is the change in slope.

In [ ]:
def ols_full(Xmat, y, names):
    n, k = Xmat.shape
    XtX_inv = np.linalg.inv(Xmat.T @ Xmat)
    b = XtX_inv @ Xmat.T @ y
    resid = y - Xmat @ b
    rss = resid @ resid
    tss = ((y - y.mean()) ** 2).sum()
    sigma2 = rss / (n - k)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    t = b / se
    p = 2 * stats.t.sf(np.abs(t), df=n - k)
    return (pd.DataFrame({"coefficient": b, "std_error": se, "t_stat": t, "p_value": p},
                         index=names),
            1 - rss / tss)

def side_fit(df, side):
    # Straight line fit of outcome on the centred score, on one side of the cutoff.
    sub = df[df["score"] < CUTOFF] if side == "below" else df[df["score"] >= CUTOFF]
    xs = sub["score"].to_numpy(float) - CUTOFF
    ys = sub["outcome"].to_numpy(float)
    Xm = np.column_stack([np.ones(len(xs)), xs])
    tab, r2 = ols_full(Xm, ys, ["intercept_at_cutoff", "slope"])
    return sub, tab, r2

def rd_pooled(df):
    xs = df["score"].to_numpy(float) - CUTOFF
    dd = (df["score"].to_numpy(float) >= CUTOFF).astype(float)
    ys = df["outcome"].to_numpy(float)
    Xm = np.column_stack([np.ones(len(xs)), dd, xs, dd * xs])
    return ols_full(Xm, ys, ["intercept", "jump_at_cutoff", "slope_below", "slope_change"])

rows = []
fits = {}
for k, df in rd.items():
    _, tab_lo, r2_lo = side_fit(df, "below")
    _, tab_hi, r2_hi = side_fit(df, "above")
    pooled, r2_pool = rd_pooled(df)
    fits[k] = {"below": tab_lo, "above": tab_hi, "pooled": pooled,
               "r2_below": r2_lo, "r2_above": r2_hi}
    rows.append({
        "dataset": k,
        "slope_before": tab_lo.loc["slope", "coefficient"],
        "t_slope_before": tab_lo.loc["slope", "t_stat"],
        "slope_after": tab_hi.loc["slope", "coefficient"],
        "t_slope_after": tab_hi.loc["slope", "t_stat"],
        "slope_change": tab_hi.loc["slope", "coefficient"] - tab_lo.loc["slope", "coefficient"],
        "jump_at_cutoff": (tab_hi.loc["intercept_at_cutoff", "coefficient"]
                           - tab_lo.loc["intercept_at_cutoff", "coefficient"]),
        "t_jump": pooled.loc["jump_at_cutoff", "t_stat"],
        "r2_below": r2_lo, "r2_above": r2_hi,
    })

rd_summary = pd.DataFrame(rows).set_index("dataset")
rd_summary

In [ ]:
for k in rd:
    print(f"===== dataset {k} =====")
    print("below the cutoff:")
    print(fits[k]["below"])
    print("\nat or above the cutoff:")
    print(fits[k]["above"])
    print("\nsingle pooled regression:")
    print(fits[k]["pooled"])
    print()

## 7. The picture

Raw points, binned averages of the score, and the fitted line on each side. The vertical gap at
80 is the estimated effect of the course.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, k in zip(axes, rd):
    df = rd[k]
    ax.scatter(df["score"], df["outcome"], s=8, alpha=0.2, color="grey")

    # binned averages make the shape visible when the outcome is 0/1
    bins = np.arange(np.floor(df["score"].min()), np.ceil(df["score"].max()) + 2, 2.0)
    df = df.assign(bin=pd.cut(df["score"], bins))
    bm = df.groupby("bin", observed=True).agg(x=("score", "mean"), y=("outcome", "mean")).dropna()
    ax.scatter(bm["x"], bm["y"], s=40, color="black", zorder=3, label="binned average")

    for side, color in [("below", "steelblue"), ("above", "crimson")]:
        sub, tab, _ = side_fit(rd[k], side)
        grid = np.linspace(sub["score"].min(), sub["score"].max(), 50)
        a = tab.loc["intercept_at_cutoff", "coefficient"]
        s = tab.loc["slope", "coefficient"]
        ax.plot(grid, a + s * (grid - CUTOFF), color=color, lw=2.5,
                label=f"{side}: slope {s:.5f}")

    ax.axvline(CUTOFF, color="black", ls="--", lw=1.5)
    jump = rd_summary.loc[k, "jump_at_cutoff"]
    ax.set_title(f"dataset {k}  -  jump at {CUTOFF}: {jump:.5f}")
    ax.set_xlabel("test score X"); ax.set_ylabel("outcome Y")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Reading off the answers

- **Q3** - the jump at the cutoff in each dataset, with its t-statistic. A jump that is positive
  and clearly separated from zero says the course helped; near zero says it did not.
- **Q4** - the dataset with a nonzero linear term is the one whose slopes on either side are far
  from zero and statistically solid, so `Y` really does trend with `X` away from the cutoff.
- **Q5** - the sign of the slope below the cutoff in that dataset.
- **Q6** - the slope after the cutoff compared with the slope before.

In [ ]:
linear_dataset = rd_summary[["t_slope_before", "t_slope_after"]].abs().min(axis=1).idxmax()

print("Q3  jump at the cutoff (effect of the course)")
for k in rd:
    j = rd_summary.loc[k, "jump_at_cutoff"]
    t = rd_summary.loc[k, "t_jump"]
    verdict = "helps" if (j > 0 and abs(t) > 2) else ("hurts" if (j < 0 and abs(t) > 2)
                                                      else "no clear effect")
    print(f"      dataset {k}: jump {j:+.6f}  (t = {t:.3f})  ->  {verdict}")

print("\nQ4  which dataset has a nonzero linear term")
for k in rd:
    print(f"      dataset {k}: slope before {rd_summary.loc[k, 'slope_before']:+.6f} "
          f"(t {rd_summary.loc[k, 't_slope_before']:7.3f}),  "
          f"slope after {rd_summary.loc[k, 'slope_after']:+.6f} "
          f"(t {rd_summary.loc[k, 't_slope_after']:7.3f})")
print(f"      ->  dataset {linear_dataset}")

sb = rd_summary.loc[linear_dataset, "slope_before"]
sa = rd_summary.loc[linear_dataset, "slope_after"]
print(f"\nQ5  in dataset {linear_dataset}, Y before the cutoff is "
      f"{'INCREASING' if sb > 0 else 'DECREASING'}  (slope {sb:+.6f})")
print(f"\nQ6  slope after ({sa:+.6f}) vs before ({sb:+.6f})  ->  "
      f"{'HIGHER' if sa > sb else 'LOWER'} after the cutoff  (change {sa - sb:+.6f})")

## 9. Robustness - only near the cutoff

The straight-line fits above use every point. Restricting to a window around 80 gives a cleaner
read of the jump if the relationship bends far from the cutoff, and it should not change the
qualitative answers.

In [ ]:
bw_rows = []
for k, df in rd.items():
    for bw in [30, 20, 15, 10, 5]:
        sub = df[(df["score"] >= CUTOFF - bw) & (df["score"] <= CUTOFF + bw)]
        if (sub["score"] < CUTOFF).sum() < 5 or (sub["score"] >= CUTOFF).sum() < 5:
            continue
        xs = sub["score"].to_numpy(float) - CUTOFF
        dd = (sub["score"].to_numpy(float) >= CUTOFF).astype(float)
        Xm = np.column_stack([np.ones(len(xs)), dd, xs, dd * xs])
        tab, _ = ols_full(Xm, sub["outcome"].to_numpy(float),
                          ["intercept", "jump", "slope_below", "slope_change"])
        bw_rows.append({"dataset": k, "bandwidth": bw, "n": len(sub),
                        "jump": tab.loc["jump", "coefficient"],
                        "t_jump": tab.loc["jump", "t_stat"],
                        "slope_below": tab.loc["slope_below", "coefficient"],
                        "slope_change": tab.loc["slope_change", "coefficient"]})

pd.DataFrame(bw_rows).set_index(["dataset", "bandwidth"])

## 10. All answers

In [ ]:
print("PART A  (homework_4.1.csv)")
print(f"  Q1  pooled Wald ratio                    : {wald_pooled:.6f}")
print(f"  Q2  averaged over {N_BINS} narrow bands of {WC} : {wald_stratified:.6f}")
print(f"      (check the sensitivity table above if this sits between two options)\n")

print("PART B  (homework_4.2.a / homework_4.2.b, cutoff 80)")
for k in rd:
    print(f"  Q3  dataset {k}: jump {rd_summary.loc[k, 'jump_at_cutoff']:+.6f} "
          f"(t {rd_summary.loc[k, 't_jump']:.3f})")
print(f"  Q4  nonzero linear term -> dataset {linear_dataset}")
print(f"  Q5  before the cutoff Y is {'increasing' if sb > 0 else 'decreasing'} "
      f"(slope {sb:+.6f})")
print(f"  Q6  slope after the cutoff is {'higher' if sa > sb else 'lower'} "
      f"({sa:+.6f} vs {sb:+.6f})")